In [202]:
import pandas as pd
import re

In [203]:
emails_df = pd.read_csv('../data/01_extracted_emails.csv')
print(emails_df.head(5))

                                    Message-ID  \
0  18782981.1075855378110.JavaMail.evans@thyme   
1  15464986.1075855378456.JavaMail.evans@thyme   
2  24216240.1075855687451.JavaMail.evans@thyme   
3  13505866.1075863688222.JavaMail.evans@thyme   
4  30922949.1075863688243.JavaMail.evans@thyme   

                                    Date                     From  \
0  Mon, 14 May 2001 16:39:00 -0700 (PDT)  phillip.allen@enron.com   
1   Fri, 4 May 2001 13:51:00 -0700 (PDT)  phillip.allen@enron.com   
2  Wed, 18 Oct 2000 03:00:00 -0700 (PDT)  phillip.allen@enron.com   
3  Mon, 23 Oct 2000 06:13:00 -0700 (PDT)  phillip.allen@enron.com   
4  Thu, 31 Aug 2000 05:07:00 -0700 (PDT)  phillip.allen@enron.com   

                        To    Subject   Cc  Mime-Version  \
0     tim.belden@enron.com        NaN  NaN           1.0   
1  john.lavorato@enron.com        Re:  NaN           1.0   
2   leah.arsdall@enron.com   Re: test  NaN           1.0   
3    randall.gay@enron.com        NaN  NaN  

In [204]:
emails_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Message-ID                 495554 non-null  object 
 1   Date                       495554 non-null  object 
 2   From                       495554 non-null  object 
 3   To                         495554 non-null  object 
 4   Subject                    478886 non-null  object 
 5   Cc                         124262 non-null  object 
 6   Mime-Version               495554 non-null  float64
 7   Content-Type               495554 non-null  object 
 8   Content-Transfer-Encoding  495554 non-null  object 
 9   Bcc                        126416 non-null  object 
 10  X-From                     495554 non-null  object 
 11  X-To                       495554 non-null  object 
 12  X-cc                       127172 non-null  object 
 13  X-bcc                      16

In [205]:
emails_df.isnull().sum()

Message-ID                    21847
Date                          21847
From                          21847
To                            21847
Subject                       38515
Cc                           393139
Mime-Version                  21847
Content-Type                  21847
Content-Transfer-Encoding     21847
Bcc                          390985
X-From                        21847
X-To                          21847
X-cc                         390229
X-bcc                        517233
X-Folder                      21847
X-Origin                      21847
X-FileName                    22394
Message-Body                  21848
dtype: int64

In [206]:
emails_df.dropna(how='all', inplace=True)
emails_df.dropna(subset=['Message-Body'], inplace=True)

In [207]:
# drop columns with lots of missing values
print(emails_df['Bcc'].isnull().mean() * 100, '% of X-bcc is empty')
print(emails_df['Cc'].isnull().mean() * 100, '% of X-cc is empty')
print(emails_df['X-bcc'].isnull().mean() * 100, '% of X-bcc is empty')
print(emails_df['X-cc'].isnull().mean() * 100, '% of X-cc is empty')

emails_df.drop(columns=['Bcc', 'Cc', 'X-bcc', 'X-cc'], inplace=True)
print('remaining cols: ', emails_df.columns)

emails_df.reset_index(drop=True, inplace=True)

74.4899132887905 % of X-bcc is empty
74.92457920747125 % of X-cc is empty
99.96609847988005 % of X-bcc is empty
74.33735644825074 % of X-cc is empty
remaining cols:  Index(['Message-ID', 'Date', 'From', 'To', 'Subject', 'Mime-Version',
       'Content-Type', 'Content-Transfer-Encoding', 'X-From', 'X-To',
       'X-Folder', 'X-Origin', 'X-FileName', 'Message-Body'],
      dtype='object')


In [208]:
# Impute with No Subject
emails_df['Subject'].fillna('No Subject', inplace=True)

In [209]:
# Drop columns that are not useful for prediction
emails_df = emails_df.drop(columns=['Message-ID', 'Mime-Version', 'Content-Type', 'Content-Transfer-Encoding', 'X-Folder', 'X-Origin', 'X-FileName'])

In [210]:
emails_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495553 entries, 0 to 495552
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   Date          495553 non-null  object
 1   From          495553 non-null  object
 2   To            495553 non-null  object
 3   Subject       495553 non-null  object
 4   X-From        495553 non-null  object
 5   X-To          495553 non-null  object
 6   Message-Body  495553 non-null  object
dtypes: object(7)
memory usage: 26.5+ MB


In [211]:
print(emails_df['Message-Body'][::16000])

0                                 Here is our forecast\n\n 
16000     in\n\n\n   \n\n\nFrom:  Bryan Hull            ...
32000     The Commission issued an order on May 8, 2001 ...
48000     Hey Dirk,\n\n Could you tell me where I would ...
64000     713 646-8525\n\n -----Original Message-----\nF...
80000      The Ron Brown Scholarships are available for ...
96000     The volume for the 24th is in error.  It is re...
112000    ---------------------- Forwarded by Drew Fossu...
128000    Hey, I'm too tired to think so I started going...
144000    Deal 457758.1 for 11/14/00, HE 7 shows 25mw @ ...
160000    Myself,  being of Mexican descent there is no ...
176000    Susan/Tana,\n\nWe have recently traded a deal ...
192000    ---------------------- Forwarded by Vince J Ka...
208000    Our proposal was accepted. Dust off your San F...
224000    I'll send it back on Tuesday.\n\n\n\n\nJose Be...
240000    Attached please find two documents for Friday'...
256000    Wait until Louise gets back.  

In [212]:
print(emails_df['Subject'][9], emails_df['Message-Body'][9])

FW: fixed forward or other Collar floor gas price terms ---------------------- Forwarded by Phillip K Allen/HOU/ECT on 10/16/2000 
01:42 PM ---------------------------


"Buckner, Buck" <buck.buckner@honeywell.com> on 10/12/2000 01:12:21 PM
To: "'Pallen@Enron.com'" <Pallen@Enron.com>
cc:  
Subject: FW: fixed forward or other Collar floor gas price terms


Phillip,

> As discussed  during our phone conversation, In a Parallon 75 microturbine
> power generation deal for a national accounts customer, I am developing a
> proposal to sell power to customer at fixed or collar/floor price. To do
> so I need a corresponding term gas price for same. Microturbine is an
> onsite generation product developed by Honeywell to generate electricity
> on customer site (degen). using natural gas. In doing so,  I need your
> best fixed price forward gas price deal for 1, 3, 5, 7 and 10 years for
> annual/seasonal supply to microturbines to generate fixed kWh for
> customer. We have the opportunity to sel

In [213]:
emails_df['Message-Body']

0                                 Here is our forecast\n\n 
1         Traveling to have a business meeting takes the...
2                            test successful.  way to go!!!
3         Randy,\n\n Can you send me a schedule of the s...
4                       Let's shoot for Tuesday at 11:45.  
                                ...                        
495548    This is a trade with OIL-SPEC-HEDGE-NG (John L...
495549    Some of my position is with the Alberta Term b...
495550    2\n\n -----Original Message-----\nFrom: \tDouc...
495551    Analyst\t\t\t\t\tRank\n\nStephane Brodeur\t\t\...
495552    i think the YMCA has a class that is for peopl...
Name: Message-Body, Length: 495553, dtype: object

In [214]:
# Find the emails that have common reply or forward patterns
# -+\s*(Original|Forwarded)
def find_reply_forward(msg_body):

    if not isinstance(msg_body, str):
        return False

    pattern = re.compile(
        r"""(?i)
        -+\s*(Original|Forwarded # forwarded
        |On\s.+wrote: # replies
        |>\s*From: # Quoted replies
        |>\s*Sent:
        |>\s*To:
        |Note:\s*forwarded\s*message\s*attached  # Forwarded note
        |Begin forwarded message
        |Message forwarded
        )""",
        re.IGNORECASE | re.VERBOSE | re.DOTALL)
    return bool(pattern.search(msg_body))

emails_df['has_reply_forward_in_msg'] = emails_df['Message-Body'].apply(find_reply_forward)


In [215]:
emails_df['has_reply_forward_in_msg'].value_counts()

False    323856
True     171697
Name: has_reply_forward_in_msg, dtype: int64

In [216]:
# Remove the emails that have reply or forward
emails_df = emails_df[emails_df['has_reply_forward_in_msg'] == False].reset_index(drop=True)
emails_df['has_reply_forward_in_msg'].value_counts()
emails_df.drop(columns=['has_reply_forward_in_msg'], inplace=True)



In [218]:
# trim the spaces
def trim_space(msg_body):
    if not isinstance(msg_body, str):
        return ''
    else:
        msg_body = msg_body.strip()
        msg_body = re.sub(r'\s+', ' ', msg_body)
        return msg_body

emails_df['Message-Body'] = emails_df['Message-Body'].apply(trim_space)

In [221]:
emails_df.describe().T

,count,unique,top,freq
Date,323856,139927,"Mon, 31 Dec 1979 16:00:00 -0800 (PST)",283
From,323856,17828,pete.davis@enron.com,9147
To,323856,40377,pete.davis@enron.com,9146
Subject,323856,104257,No Subject,14837
X-From,323856,23884,Enron Announcements,8525
X-To,323856,49187,pete.davis@enron.com,5334
Message-Body,323856,146805,We've updated the Merger Q&A document on our E...,110


In [ ]:
emails_df.to_csv('../data/02_emails_handled_missing_reply_forward.csv', index=False)